Every request Claude processes gets tokenized, embedded, and analyzed from scratch. When you send follow-up messages with the same large document or system prompt, all that preprocessing repeats — wasted work.

### How Caching Fixes It

First request writes to the cache. Follow-up requests read from it instead of reprocessing. Cache lives for **1 hour**.

In [ ]:
import anthropic
from dotenv import load_dotenv

load_dotenv()
client = anthropic.Anthropic()
model = "claude-opus-4-6"

In [ ]:
with open("C:\Code\exp\sum_up.md", "r") as f:
    document = f.read()

In [ ]:
def ask_with_cache(question):
    response = client.messages.create(
        model=model,
        max_tokens=1024,
        system=[
            {
                "type": "text",
                "text": "You are a helpful assistant that answers questions about the provided document.",
            },
            {
                "type": "text",
                "text": document,
                "cache_control": {"type": "ephemeral"}  # mark this for caching
            }
        ],
        messages=[{"role": "user", "content": question}]
    )

    usage = response.usage
    print(f"Input tokens: {usage.input_tokens}")
    print(f"Cache write tokens: {usage.cache_creation_input_tokens}")
    print(f"Cache read tokens: {usage.cache_read_input_tokens}")
    print()
    return response.content[0].text

In [ ]:
print(ask_with_cache("What is the main topic of this document?"))
print(ask_with_cache("What are the key findings?"))
print(ask_with_cache("Summarize the conclusion."))